# Llama-4-Scout-17B vLLM Text Generation Demo

Three protocols: REST Generate, gRPC Streaming, REST Generate Stream

**Prerequisites:**
- Model weights downloaded via `scripts/download/download_llama4scout.py`
- Triton running with the vLLM backend: `docker compose --profile vllm up -d`
- Model loaded: `curl -X POST $TRITON_REST_URL/v2/repository/models/llama4scout-vllm/load`

**Note:** The vLLM backend uses a decoupled (streaming) transaction policy, so the standard
Triton `/infer` HTTP endpoint is not supported. Use `/generate` for HTTP and streaming infer for gRPC.

In [ ]:
import os
import json
import time
import numpy as np
import requests
import tritonclient.grpc as grpcclient

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"

os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")
MODEL_NAME = "llama4scout-vllm"

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else {}

print(f"Namespace:  {NAMESPACE}")
print(f"gRPC URL:   {GRPC_URL}")
print(f"REST URL:   {REST_URL}")
print(f"Model:      {MODEL_NAME}")
print(f"Auth:       {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Check model status
# =============================================================================
resp = requests.get(f"{REST_URL}/v2/health/ready", headers=http_headers)
print(f"Triton ready: {resp.status_code == 200}")

resp = requests.get(f"{REST_URL}/v2/models/{MODEL_NAME}/ready", headers=http_headers)
if resp.status_code == 200:
    print(f"Model '{MODEL_NAME}': ready")
else:
    print(f"Model '{MODEL_NAME}': not loaded (status {resp.status_code})")
    print("Load it with:")
    print(f"  curl -X POST {REST_URL}/v2/repository/models/{MODEL_NAME}/load")

In [ ]:
# =============================================================================
# Prepare inputs
# =============================================================================
prompt = "Explain the Mixture-of-Experts architecture used in Llama 4 Scout."
max_tokens = 256
temperature = 0.7
top_p = 0.9

print(f"Prompt:     {prompt}")
print(f"Max tokens: {max_tokens}")
print(f"Temp:       {temperature}")

In [ ]:
# =============================================================================
# 1. REST - /generate endpoint (non-streaming)
# vLLM decoupled models do not support the standard /infer HTTP endpoint.
# Use /v2/models/{model}/generate instead.
# =============================================================================
payload = {
    "text_input": prompt,
    "parameters": {
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "stream": False,
    }
}

start = time.time()
resp = requests.post(
    f"{REST_URL}/v2/models/{MODEL_NAME}/generate",
    json=payload,
    headers=http_headers,
)
resp.raise_for_status()
t_rest = (time.time() - start) * 1000
text_rest = resp.json()["text_output"]

print(f"REST (generate): {t_rest:.0f} ms")
print(f"\nResponse:\n{text_rest}")

In [ ]:
# =============================================================================
# 2. gRPC - async streaming infer (required for decoupled models)
# Uses tritonclient.grpc.aio which works natively in Jupyter's event loop.
# =============================================================================
import tritonclient.grpc.aio as aio_grpcclient

async def infer_grpc():
    client = aio_grpcclient.InferenceServerClient(url=GRPC_URL)

    inp_text = aio_grpcclient.InferInput("text_input", [1], "BYTES")
    inp_text.set_data_from_numpy(np.array([prompt], dtype=np.object_))

    inp_params = aio_grpcclient.InferInput("sampling_parameters", [1], "BYTES")
    inp_params.set_data_from_numpy(np.array(
        [json.dumps({"temperature": temperature, "top_p": top_p, "max_tokens": max_tokens})],
        dtype=np.object_
    ))

    inp_stream = aio_grpcclient.InferInput("stream", [1], "BOOL")
    inp_stream.set_data_from_numpy(np.array([False], dtype=bool))

    outputs = [aio_grpcclient.InferRequestedOutput("text_output")]

    async def request_iterator():
        yield {
            "model_name": MODEL_NAME,
            "inputs": [inp_text, inp_params, inp_stream],
            "outputs": outputs,
        }

    result_text = ""
    async for result, error in client.stream_infer(
        request_iterator(),
        headers=grpc_headers,
    ):
        if error:
            raise error
        token = result.as_numpy("text_output")[0]
        if isinstance(token, bytes):
            token = token.decode("utf-8")
        result_text += token

    await client.close()
    return result_text

start = time.time()
text_grpc = await infer_grpc()
t_grpc = (time.time() - start) * 1000

print(f"gRPC (streaming): {t_grpc:.0f} ms")
print(f"\nResponse:\n{text_grpc}")

In [ ]:
# =============================================================================
# 3. REST - /generate_stream endpoint (SSE streaming)
# =============================================================================
payload_stream = {
    "text_input": prompt,
    "parameters": {
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "stream": True,
    }
}

start = time.time()
text_stream = ""
with requests.post(
    f"{REST_URL}/v2/models/{MODEL_NAME}/generate_stream",
    json=payload_stream,
    headers=http_headers,
    stream=True,
) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():
        if line:
            data = line.decode("utf-8").removeprefix("data: ")
            chunk = json.loads(data)
            text_stream += chunk.get("text_output", "")
t_stream = (time.time() - start) * 1000

print(f"REST (generate_stream): {t_stream:.0f} ms")
print(f"\nResponse:\n{text_stream}")

In [ ]:
# =============================================================================
# Results summary
# =============================================================================
print(f"{'Protocol':<25} {'Latency':>10}")
print("-" * 37)
print(f"{'REST /generate':<25} {t_rest:>9.0f}ms")
print(f"{'gRPC streaming':<25} {t_grpc:>9.0f}ms")
print(f"{'REST /generate_stream':<25} {t_stream:>9.0f}ms")